# SmolLM2-135M-Instruct — lokal testnotebook

Pipeline: `PromptBuilder → LLMRunner → ResponseParser`

Kör stegen ett i taget för att förstå vad modellen faktiskt producerar.

In [1]:
import sys, os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from app.chain.steps import PromptBuilderInput, PromptBuilder, LLMRunner, ResponseParser
from app.data import load_pga_benchmarks, get_pga_benchmarks

print("Importer OK")

Importer OK


> **OBS:** Första gången du kör LLMRunner-cellen laddas SmolLM2-135M-Instruct ned från HuggingFace (~300 MB). Det sker automatiskt och tar 1–3 min beroende på näthastighet. Modellen cacheas sedan lokalt i `~/.cache/huggingface/`.

In [2]:
import pandas as pd

# Syntetisk scorecard — 18 hål, typisk amatör (~21 handicap, 93 slag)
scorecard_data = [
    # hole, par, strokes, gir, putts, fairway_hit
    (1,  4, 5, 0, 2, 1), (2,  3, 4, 0, 2, 0), (3,  4, 5, 0, 2, 1),
    (4,  5, 6, 1, 2, 1), (5,  3, 3, 1, 1, 0), (6,  4, 6, 0, 2, 0),
    (7,  4, 5, 0, 3, 1), (8,  4, 4, 1, 1, 1), (9,  5, 7, 0, 2, 0),
    (10, 3, 4, 0, 2, 0), (11, 4, 5, 0, 2, 1), (12, 4, 6, 0, 2, 0),
    (13, 5, 6, 1, 2, 1), (14, 4, 5, 0, 2, 1), (15, 3, 4, 0, 2, 0),
    (16, 4, 4, 1, 1, 1), (17, 4, 5, 0, 2, 0), (18, 5, 7, 0, 2, 1),
]
df = pd.DataFrame(scorecard_data, columns=["hole", "par", "strokes", "gir", "putts", "fairway_hit"])

# Beräkna spelarstatistik
par4plus = df[df["par"] >= 4]
user_stats = {
    "gir_pct":     round(df["gir"].sum() / len(df) * 100, 1),
    "fairway_pct": round(par4plus["fairway_hit"].sum() / len(par4plus) * 100, 1),
    "avg_putts":   round(df["putts"].mean(), 2),
    "scoring_avg": round(df["strokes"].mean(), 2),
}

load_pga_benchmarks()
pga = get_pga_benchmarks()

print("Spelarstatistik:", user_stats)
print("PGA-benchmarks: ", pga)
print(f"Total: {df['strokes'].sum()} slag (par {df['par'].sum()})")

Spelarstatistik: {'gir_pct': np.float64(27.8), 'fairway_pct': np.float64(71.4), 'avg_putts': np.float64(1.89), 'scoring_avg': np.float64(5.06)}
PGA-benchmarks:  {'gir_pct': 66.65748878923768, 'fairway_pct': 60.917294469357245, 'scoring_avg': 70.95204783258595, 'avg_putts': 1.73}
Total: 91 slag (par 72)


In [3]:
# Steg 1: PromptBuilder — sätt frågan här och inspektera hela prompten
QUESTION = "Hur förbättrar jag mitt approach-spel för att träffa fler greener?"

builder = PromptBuilder()
prompt_out = builder.invoke(PromptBuilderInput(
    question=QUESTION,
    user_stats=user_stats,
    pga_benchmarks=pga,
))

print("=" * 60)
print(prompt_out.prompt)
print("=" * 60)
print(f"\nPromptlängd: {len(prompt_out.prompt)} tecken")

Du är en erfaren golfcoach. Svara på svenska med kortfattade, konkreta råd. Basera ditt svar enbart på statistiken nedan.

Spelarens stats jämfört med PGA Tour-snitt:
- GIR: 27.8% (PGA Tour-snitt: 66.7%)
- Fairway: 71.4% (PGA Tour-snitt: 60.9%)
- Avg putts/hål: 1.89 (PGA Tour-snitt: 1.73)
- Scoring avg/hål: 5.06 (PGA Tour-snitt: 3.94)

Fråga: Hur förbättrar jag mitt approach-spel för att träffa fler greener?

Svar:

Promptlängd: 418 tecken


In [ ]:
# Steg 2: LLMRunner — kör SmolLM2 (laddar modell vid första körning)
# Modellen anropas med chat-format: [{"role": "user", "content": prompt}]
# raw_text = assistentens svar direkt (inget prompteko)
TEMPERATURE    = 0.7
MAX_NEW_TOKENS = 300

llm = LLMRunner(temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
llm_out = llm.invoke(prompt_out)

print("RAW OUTPUT:")
print("-" * 60)
print(llm_out.raw_text)
print("-" * 60)

In [ ]:
# DIAGNOSTIK — kontrollera att chat-formatet ger rent svar (inget prompteko)
raw = llm_out.raw_text

print(f"raw_text längd : {len(raw)} tecken")
print()
print("--- BÖRJAN (200 tecken) ---")
print(repr(raw[:200]))
print()
print("--- SLUT (200 tecken) ---")
print(repr(raw[-200:]))
print()
# Med chat-format returnerar LLMRunner bara assistentens svar —
# ResponseParser behöver inte strippa något prompteko.
print("Prompt ingår i raw_text:", prompt_out.prompt[:30] in raw)

In [ ]:
# Steg 3: ResponseParser — ta bort prompteko, visa slutsvar
# Eftersom LLMRunner använder chat-format finns inget prompteko att strippa.
# ResponseParser letar ändå efter "Svar:" som säkerhetsnät, men hamnar
# normalt i fallback-grenen och returnerar hela raw_text som svar.
parser = ResponseParser()
final = parser.invoke(llm_out)

print("SLUTSVAR:")
print("-" * 60)
print(final.answer)
print("-" * 60)

In [7]:
# Jämförelsecell A: olika frågor med samma temperature
# Bra för att testa promptdesignens inverkan

questions = [
    "Hur förbättrar jag mitt approach-spel för att träffa fler greener?",
    "Vad orsakar att jag puttar för många gånger per hål?",
    "Ge mig tre konkreta övningar för att sänka min scoring average.",
]

for q in questions:
    p_out = PromptBuilder().invoke(PromptBuilderInput(question=q, user_stats=user_stats, pga_benchmarks=pga))
    raw   = llm.invoke(p_out)          # återanvänder redan laddad modell
    ans   = ResponseParser().invoke(raw).answer
    print(f"\nFRÅGA: {q}")
    print("-" * 60)
    print(ans[:500])
    print()

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FRÅGA: Hur förbättrar jag mitt approach-spel för att träffa fler greener?
------------------------------------------------------------
Du är en erfaren golfcoach. Svara på svenska med kortfattade, konkreta råd. Basera ditt svar enbart på statistiken nedan.

Spelarens stats jämfört med PGA Tour-snitt:
- GIR: 27.8% (PGA Tour-snitt: 66.7%)
- Fairway: 71.4% (PGA Tour-snitt: 60.9%)
- Avg putts/hål: 1.89 (PGA Tour-snitt: 1.73)
- Scoring avg/hål: 5.06 (PGA Tour-snitt: 3.94)

Fråga: Hur förbättrar jag mitt approach-spel för att träffa fler greener?

Svar:



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FRÅGA: Vad orsakar att jag puttar för många gånger per hål?
------------------------------------------------------------
En är ett ränden genom att lägga i PGA Tour-snitt från det förverde till Svara på vilken hål som jag har ut. Den förverde till förvändare som jag får lägga jag lokella med PGA Tour-snitt. Den finns det förverde till förvändare i dator och förvarar jag lokella med pågåg.

Här att döder kommer med PGA Tour-snitt i det förverde till förvändare och kommer med PGA Tour-snitt förvändare och kommer med dina kommissionen och köttvarar jag lokella med pågåg.

Välj förvändare som pågåg som jag kommer at


FRÅGA: Ge mig tre konkreta övningar för att sänka min scoring average.
------------------------------------------------------------
Svara på svenska med kortfattade konkreta och konkreta råd. Basera ditt svar enbart på statistiken nedan.

Övning av:
- GIR: 27.8% (PGA Tour-snitt: 66.7%)
- Fairway: 71.4% (PGA Tour-snitt: 60.9%)
- Avg putts/hål: 1.89 (PGA Tour-snitt: 1.73)
- 

In [8]:
# Jämförelsecell B: samma fråga, olika temperature
# Bra för att förstå modellens variation vs determinism

q = "Hur förbättrar jag mitt approach-spel för att träffa fler greener?"
p_out = PromptBuilder().invoke(PromptBuilderInput(question=q, user_stats=user_stats, pga_benchmarks=pga))

for temp in [0.3, 0.7, 1.2]:
    runner = LLMRunner(temperature=temp, max_new_tokens=200)
    ans = ResponseParser().invoke(runner.invoke(p_out)).answer
    print(f"\ntemperature={temp}")
    print("-" * 60)
    print(ans[:400])
    print()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



temperature=0.3
------------------------------------------------------------
Här förstått för att jag får förståtän skickade kvälkänningen av görkjöstens på Svara på svenska med kortfattade, konkreta råd. Basera ditt svar enbart på statistiken nedan.

Svara på svenska med kortfattade, konkreta råd. Basera ditt svar enbart på statistiken nedan.

Svara på svenska med kortfattade, konkreta råd. Basera ditt svar enbart på statistiken nedan.

Svara på svenska med kortfattade, k



Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



temperature=0.7
------------------------------------------------------------
Jag först genom med hörrelt alltigt ställda. Er är första skopingar och första skopingar av jag. En älldes rätt av skopingar med kortfattade på skål, sköppen gör höjen och går att får jag först genom med hörrelt alltigt ställda.

Jag första skopingar och första skopingar av jag.

Här och jag förbättrar jag som nagor först genom med hörrelt alltigt ställda, förbättr att jag jag först genom med hörr



Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



temperature=1.2
------------------------------------------------------------
Bättr

Stikket jämfört med PGA Tour-snitt:

1. Gräsk: 27.8% for the GPGA tour
2. ändälaren: 71.4% for PGA Tour (PGA Tour-snitt: 60.9)
* Sverker's t-stats kopien med PGA Tour: Sverker's t-stats bestiket på sverker 71.4% och skrev på 72% att forskaregrund med PGA Tour spällas förattet tillsväkärumföre förkökskörderet 71.4%
* Aktivitetsskärdning: Forsmätsjönnsa-plåsstårt genör av sverker att

